# 12.3 Test enrichment

The test inputs are built by the base-then-enrich pipeline run once on this agent's domain. Twenty labeled complaint cases are the base; three presentation factors (clarity, entity_aliasing, reasoning_cue) vary *how* a complaint is presented without changing its correct handling. The product is the scenario suite the rest of the chapter runs on: one balanced run, scored many ways.

**Reference (run separately):** the suite is composed by `gt.compose`, with the seed case entering as a balanced blocking factor and the three presentation factors space-filled by the Sobol-plus-refine generator. This is the heavy construction that produced the numbers below; it is shown for reference and is not executed here.

```python
import gmstest as gt
from functools import partial
from gmstest.compose import graphdoe_design

cat   = gt.Catalog.load()
items = gt.SeedCaseSource(cases).items()                  # 20 labeled cases
suite = gt.resolve(cat, ['conditional_rule'],
                   ['clarity', 'entity_aliasing', 'reasoning_cue'], mode='embedded')
scns  = gt.compose(suite, items, n_runs=120, seed=42,
                   design_fn=partial(graphdoe_design, method='sobol+refine'),
                   balance_base=True, level_overrides=overrides)
```

**What the next cell does** — load the pinned suite artifacts:

1. **Find the repo root.** Walk up from `Path.cwd()` until `data/capstone_run.json` exists, so the notebook runs from any directory.
2. **Read the JSON.** Load `data/capstone_run.json` into `D` and `data/capstone_companions.json` into `C` with `json.loads`.

In [ ]:
import json
from pathlib import Path

root = Path.cwd().parent / 'code'
while not (root / 'data' / 'capstone_run.json').exists() and root != root.parent:
    root = root.parent
D = json.loads((root / 'data' / 'capstone_run.json').read_text())
C = json.loads((root / 'data' / 'capstone_companions.json').read_text())
print('loaded capstone_run.json and capstone_companions.json')

**What the next cell does** — report the suite's balance:

1. **Pull the counts.** Read `D['n_runs']`, `D['seed_balance']` (runs per base case) and `D['clarity_balance']` (runs per clarity level).
2. **Print the size and blocking.** Show `n_runs`, then the number of base cases in `seed_balance` with the distinct counts present and their min and max.
3. **Print clarity coverage.** Loop over `clear`, `ambiguous`, `misleading`, print each `clarity_balance[level]` and the total.

In [ ]:
n_runs = D['n_runs']
seed_balance = D['seed_balance']
clarity_balance = D['clarity_balance']

print(f'scenarios in the suite: {n_runs}')
print()
print('seed-case blocking (runs per base case):')
counts = sorted(set(seed_balance.values()))
print(f'  {len(seed_balance)} cases, counts present = {counts}  '
      f'(min {min(seed_balance.values())}, max {max(seed_balance.values())})')
print()
print('clarity factor coverage:')
for level in ('clear', 'ambiguous', 'misleading'):
    print(f'  {level:11s} {clarity_balance[level]:3d}')
print(f'  {"total":11s} {sum(clarity_balance.values()):3d}')

The result is 120 scenarios: the blocking holds the seed case exactly balanced (six runs each across all twenty cases), while the presentation factors are near-uniformly covered by the space-filling generator. Because every case is exercised equally and the factors are spread by design rather than hand-picked, the factor attribution later in the chapter reads a controlled design. This one balanced suite drives the outcome, trajectory, component and groundedness checks.

**What the next cell does** — assert the suite matches the design:

1. **Check the size.** Assert `n_runs == 120`.
2. **Check the blocking.** Assert `seed_balance` has 20 cases, every count is exactly 6, and the counts sum to `n_runs`.
3. **Check the factor.** Assert `clarity_balance` covers `clear`, `ambiguous`, `misleading` and sums to `n_runs`.

In [ ]:
# the suite has the expected size
assert n_runs == 120, n_runs

# seed-case blocking is exactly balanced: every base case appears the same number of times
assert len(seed_balance) == 20, len(seed_balance)
assert set(seed_balance.values()) == {6}, seed_balance
assert sum(seed_balance.values()) == n_runs

# the presentation factor covers all three levels and sums to the suite size
assert set(clarity_balance) == {'clear', 'ambiguous', 'misleading'}, clarity_balance
assert sum(clarity_balance.values()) == n_runs, clarity_balance

print('self-check passed')